In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tqdm.auto import tqdm
import sacrebleu
from comet import download_model, load_from_checkpoint
import os

from model_encoder_decoder import Transformer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT_PATH = "checkpoint_encoder_decoder.pth"
TEST_DATA_PATH = "../data/test_data.tsv"
TOKENIZER_PATH = "tokenizer_encoder_decoder.json"

D_MODEL = 256
N_LAYERS = 4           
N_HEAD = 8
MAX_LEN = 1000

class TestDataset(Dataset):
    def __init__(self, path, tokenizer_path):
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.pairs = []
        
        self.sos_id = self.tokenizer.token_to_id("<sos>")
        self.eos_id = self.tokenizer.token_to_id("<eos>")
        
        print(f"Reading test data from {path}...")
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 4:
                    self.pairs.append((parts[1], parts[3])) # (English, Russian)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_text, tgt_text = self.pairs[idx]
        src_ids = self.tokenizer.encode(src_text).ids
        src_tensor = torch.tensor([self.sos_id] + src_ids + [self.eos_id], dtype=torch.long)
        return src_tensor, src_text, tgt_text

def collate_test(batch):
    src_tensors, src_texts, tgt_texts = zip(*batch)
    pad_id = 0
    src_padded = torch.nn.utils.rnn.pad_sequence(src_tensors, padding_value=pad_id, batch_first=True)
    return src_padded, src_texts, tgt_texts

def generate_translations(model, loader, tokenizer, device):
    model.eval()
    generated_texts = []
    reference_texts = []
    source_texts = []
    
    sos_id = tokenizer.token_to_id("<sos>")
    eos_id = tokenizer.token_to_id("<eos>")
    
    print("Generating translations with EOS check...")
    with torch.no_grad():
        for src, src_txt, tgt_txt in tqdm(loader):
            src = src.to(device)
            batch_size = src.size(0)
            
            decoder_input = torch.full((batch_size, 1), sos_id, dtype=torch.long, device=device)
            
            finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
            
            for _ in range(100):
                logits = model(src, decoder_input)
                next_token = logits[:, -1, :].argmax(dim=-1).unsqueeze(1)
                
                decoder_input = torch.cat([decoder_input, next_token], dim=1)
                
                is_eos = (next_token.squeeze(1) == eos_id)
                finished = finished | is_eos
                
                if finished.all():
                    break
            
            decoded_batch = []
            for i in range(batch_size):
                seq = decoder_input[i].tolist()
                
                try:
                    eos_index = seq.index(eos_id)
                    seq = seq[:eos_index]
                except ValueError:
                    pass
                
                text = tokenizer.decode(seq, skip_special_tokens=True)
                decoded_batch.append(text)
            
            generated_texts.extend(decoded_batch)
            reference_texts.extend(tgt_txt)
            source_texts.extend(src_txt)
            
    return source_texts, generated_texts, reference_texts

if __name__ == "__main__":
    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
    vocab_size = tokenizer.get_vocab_size()
    pad_idx = tokenizer.token_to_id("<pad>")
    
    test_dataset = TestDataset(TEST_DATA_PATH, TOKENIZER_PATH)
    test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_test)

    print("Loading model...")
    model = Transformer(
        vocab_size_seq=vocab_size,
        vocab_size_target=vocab_size,
        d_model=D_MODEL,
        n_layer=N_LAYERS,
        n_head=N_HEAD,
        d_head=D_MODEL // N_HEAD,
        d_ff=D_MODEL * 4,
        max_len=MAX_LEN,
        dropout=0.1,
        pad_idx=pad_idx
    ).to(DEVICE)
    
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Model loaded from epoch {checkpoint['epoch']}")

    sources, hypotheses, references = generate_translations(model, test_loader, tokenizer, DEVICE)

    print("\nCalculating metrics...")
    
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    print(f"BLEU: {bleu.score:.2f}")
    
    chrf = sacrebleu.corpus_chrf(hypotheses, [references], word_order=2)
    print(f"ChrF++: {chrf.score:.2f}")

    with open("results_comparison.txt", "w", encoding="utf-8") as f:
        for src, hyp, ref in zip(sources, hypotheses, references):
            f.write(f"SRC: {src}\nREF: {ref}\nHYP: {hyp}\n{'-'*20}\n")
    print("Translations saved to results_comparison.txt")

    try:
        print("Calculating COMET (this might take a while)...")
        model_path = download_model("Unbabel/wmt22-comet-da")
        comet_model = load_from_checkpoint(model_path)
        data = [{"src": s, "mt": h, "ref": r} for s, h, r in zip(sources, hypotheses, references)]
        comet_score = comet_model.predict(data, batch_size=16, gpus=1)
        print(f"COMET: {comet_score.system_score:.4f}")
    except Exception as e:
        print(f"COMET calculation failed (probably OOM or internet): {e}")


/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /usr/local/lib/python3.10/dist-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2025-11-26 19:34:12.754378: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flag

Reading test data from data/test_data.tsv...
Loading model...
Model loaded from epoch 19
Generating translations with EOS check...


100%|██████████| 938/938 [04:08<00:00,  3.78it/s]


Calculating metrics...



That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


BLEU: 39.42
ChrF++: 60.72
Translations saved to results_comparison.txt
Calculating COMET (this might take a while)...


Fetching 5 files: 100%|██████████| 5/5 [00:09<00:00,  1.99s/it]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../tmp/xdg_cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/jupyter/.local/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100%|██████████| 

COMET: 0.8308


In [3]:
import json
from datetime import datetime

# Собираем словарь с результатами
metrics_log = {
    "model_type": "Encoder-Decoder Transformer",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "dataset_file": TEST_DATA_PATH,
    "metrics": {
        "BLEU": round(bleu.score, 2),
        "ChrF++": round(chrf.score, 2),
        # Проверяем, существует ли переменная comet_score (вдруг расчет упал)
        "COMET": round(comet_score.system_score, 4) if 'comet_score' in locals() else None
    },
    "config": {
        "d_model": D_MODEL,
        "n_layers": N_LAYERS,
        "n_head": N_HEAD
    }
}

# Имя файла, как вы просили
filename = "fancy_metrics_encoder_decoder.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(metrics_log, f, indent=4, ensure_ascii=False)

print(f"✅ Метрики успешно сохранены в файл: {filename}")
print("-" * 30)
print(json.dumps(metrics_log, indent=4, ensure_ascii=False))


✅ Метрики успешно сохранены в файл: fancy_metrics_encoder_decoder.json
------------------------------
{
    "model_type": "Encoder-Decoder Transformer",
    "timestamp": "2025-11-26 19:42:19",
    "dataset_file": "data/test_data.tsv",
    "metrics": {
        "BLEU": 39.42,
        "ChrF++": 60.72,
        "COMET": 0.8308
    },
    "config": {
        "d_model": 256,
        "n_layers": 4,
        "n_head": 8
    }
}
